In [1]:
!pip install -q gradio

In [3]:
from google.colab import files

files.upload()

Saving emotion_distilbert.zip to emotion_distilbert.zip


In [8]:
from google.colab import files

files.upload()

Saving label_encoder.pkl to label_encoder.pkl


{'label_encoder.pkl': b'\x80\x04\x95\x1c\x01\x00\x00\x00\x00\x00\x00\x8c\x1csklearn.preprocessing._label\x94\x8c\x0cLabelEncoder\x94\x93\x94)\x81\x94}\x94(\x8c\x08classes_\x94\x8c\x16numpy._core.multiarray\x94\x8c\x0c_reconstruct\x94\x93\x94\x8c\x05numpy\x94\x8c\x07ndarray\x94\x93\x94K\x00\x85\x94C\x01b\x94\x87\x94R\x94(K\x01K\x06\x85\x94h\t\x8c\x05dtype\x94\x93\x94\x8c\x02O8\x94\x89\x88\x87\x94R\x94(K\x03\x8c\x01|\x94NNNJ\xff\xff\xff\xffJ\xff\xff\xff\xffK?t\x94b\x89]\x94(\x8c\x05anger\x94\x8c\x07disgust\x94\x8c\x04fear\x94\x8c\x03joy\x94\x8c\x07sadness\x94\x8c\x08surprise\x94et\x94b\x8c\x10_sklearn_version\x94\x8c\x051.6.1\x94ub.'}

In [4]:
!unzip emotion_distilbert.zip

Archive:  emotion_distilbert.zip
   creating: emotion_distilbert/
  inflating: emotion_distilbert/config.json  
  inflating: emotion_distilbert/model.safetensors  
  inflating: emotion_distilbert/tokenizer_config.json  
  inflating: emotion_distilbert/tokenizer.json  
  inflating: emotion_distilbert/training_args.bin  


In [9]:
!ls

emotion_distilbert  emotion_distilbert.zip  label_encoder.pkl  sample_data


In [16]:
import gradio as gr
import plotly.express as px
import pandas as pd
import torch
import numpy as np


history_data = []


def predict_emotion(text):

    if not text.strip():
        return "No text provided", None, pd.DataFrame(history_data)


    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )


    with torch.no_grad():
        outputs = model(**inputs)


    probs = torch.softmax(
        outputs.logits,
        dim=1
    )[0].numpy()


    prediction = label_encoder.classes_[
        np.argmax(probs)
    ]


    confidence = float(
        np.max(probs)
    )


    confidence_df = pd.DataFrame({

        "Emotion": label_encoder.classes_,

        "Confidence": probs

    })


    fig = px.bar(

        confidence_df,

        x="Emotion",

        y="Confidence",

        text=confidence_df["Confidence"].apply(
            lambda x: f"{x:.1%}"
        ),

        title="Emotion Confidence Distribution"

    )


    fig.update_layout(

        yaxis=dict(
            range=[0,1],
            title="Confidence"
        ),

        xaxis_title="Emotion",

        height=400,

        template="plotly_white"

    )


    history_data.append({

        "Text": text,

        "Prediction": prediction.capitalize(),

        "Confidence": f"{confidence:.2%}"

    })


    history_df = pd.DataFrame(history_data)


    return (

        prediction.capitalize(),

        fig,

        history_df

    )



css = """

.gradio-container {

    max-width: 1100px !important;

}


h1 {

    text-align:center;

}


button {

    font-size:18px !important;

}

"""


with gr.Blocks(

    theme=gr.themes.Glass(),

    css=css

) as demo:


    gr.Markdown(
    """
    <h1>🧠 Emotion Intelligence System</h1>

    <center>

    ### Fine-Tuned DistilBERT Emotion Classifier

    Detect emotions from text using Transformer NLP

    <br>

    Dataset: GoEmotions

    <br>

    Classes:
    Joy | Anger | Sadness | Fear | Surprise | Disgust

    </center>

    ---
    """
    )


    gr.Markdown(
    """
    ✍️ Enter any sentence and let the model analyze the emotion.
    """
    )


    text_input = gr.Textbox(

        label="📝 Your Text",

        placeholder=
        "Example: I am really excited about this project!",

        lines=5

    )


    with gr.Row():

        predict_btn = gr.Button(

            "🚀 Analyze Emotion",

            variant="primary"

        )


        clear_btn = gr.ClearButton(

            components=[
                text_input
            ]

        )


    gr.Markdown(
    """
    ## 🎯 Prediction Result
    """
    )


    with gr.Row():

        prediction_output = gr.Textbox(

            label="Dominant Emotion"

        )


        chart_output = gr.Plot(

            label="Confidence Scores"

        )


    gr.Markdown(
    """
    ---

    ## 🔥 Try Examples

    """
    )


    gr.Examples(

        examples=[

            ["I got amazing news today, I am so happy"],

            ["I hate this situation, everything is annoying"],

            ["I am scared and worried about my future"],

            ["I can't believe this happened, what a surprise"],

            ["I feel lonely and very sad"],

            ["This food tastes disgusting"]

        ],

        inputs=text_input

    )


    gr.Markdown(
    """
    ---

    ## 📜 Prediction History

    """
    )


    history_table = gr.Dataframe(

        headers=[

            "Text",

            "Prediction",

            "Confidence"

        ],

        label="Previous Predictions",

        interactive=False

    )


    gr.Markdown(
    """
    ---

    ## 🤖 Model Information

    - Architecture: DistilBERT
    - Dataset: GoEmotions
    - Task: Multi-Class Emotion Classification
    - Framework: HuggingFace Transformers

    """
    )


    predict_btn.click(

        fn=predict_emotion,

        inputs=text_input,

        outputs=[

            prediction_output,

            chart_output,

            history_table

        ]

    )


demo.launch()

/tmp/ipykernel_1696/1538038124.py:139: UserWarning:

The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.



It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9bf9710ccb9acbfb59.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
